# 10 - Qwen3:14B Inference

## Objective

Generate grounded Oracle ERP answers using retrieved context from both RAG pipelines.

### Pipeline 1

Recursive Chunking
→ MiniLM
→ ChromaDB
→ Qwen3:14B

### Pipeline 2

Section-Based Chunking
→ BGE-M3
→ Native FAISS
→ Qwen3:14B

## Outcome

Compare the quality of responses generated by Qwen3:14B using different retrieval strategies.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Load Source Documents

In [2]:
from src.config import DOCS_PATH
from src.loaders.document_loader import DocumentLoader

loader = DocumentLoader(DOCS_PATH)

documents = loader.load()

print(
    f"Source Pages: {len(documents):,}"
)

Starting Document Loading...

Loading PDF : iSupplier_portal.pdf

Loading PDF : Purchasing_manual.pdf


DOCUMENT LOADING SUMMARY
PDF Files Loaded   : 2
DOCX Files Loaded  : 0
XLSX Files Loaded  : 0
Total Documents    : 1502
Source Pages: 1,502


## Detect Hardware

In [3]:
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Device: {device}"
)

if torch.cuda.is_available():

    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

Device: cuda
GPU: NVIDIA RTX A5000 Laptop GPU


## Load Embedding Models

In [4]:
from langchain_huggingface import (
    HuggingFaceEmbeddings
)

minilm_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": device
    }
)

bgem3_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": device
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Load Existing ChromaDB Index

In [ ]:
from langchain_chroma import Chroma

chroma_db = Chroma(
    persist_directory=
    "../vectorstores/chroma_minilm",
    embedding_function=minilm_model
)

print(
    "ChromaDB Loaded"
)

## Load Existing FAISS Index

In [ ]:
import faiss

faiss_index = faiss.read_index(
    "../vectorstores/faiss_bgem3/oracle_bgem3.index"
)

print(
    f"FAISS Vectors: {faiss_index.ntotal:,}"
)

## Rebuild Section-Based Chunks

Required to map retrieved FAISS vectors back to source text.

In [ ]:
from langchain_core.documents import Document

In [ ]:
def create_section_chunks(
    documents,
    target_size=2000
):

    section_chunks = []

    current_text = ""

    current_metadata = None

    for doc in documents:

        page_text = doc.page_content.strip()

        if not page_text:
            continue

        if current_metadata is None:
            current_metadata = doc.metadata.copy()

        current_text += "\n\n" + page_text

        if len(current_text) >= target_size:

            section_chunks.append(
                Document(
                    page_content=current_text.strip(),
                    metadata=current_metadata
                )
            )

            current_text = ""
            current_metadata = None

    if current_text:

        section_chunks.append(
            Document(
                page_content=current_text.strip(),
                metadata=current_metadata
            )
        )

    return section_chunks

In [ ]:
section_chunks = create_section_chunks(
    documents
)

print(
    f"Section Chunks: {len(section_chunks):,}"
)

## Load Qwen3:14B From Ollama

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:14b"
)

print(
    "Qwen3:14B Ready"
)

## Interactive Question Loop

Ask Oracle ERP questions and compare answers generated by both retrieval pipelines.

In [ ]:
import numpy as np

invalid_attempts = 0

while True:

    question = input(
        "\nEnter your Oracle ERP question: "
    ).strip()

    if not question:

        print(
            "Please enter a valid question."
        )

        continue

    print("\n" + "=" * 100)

    print(
        f"QUESTION: {question}"
    )

    print("=" * 100)

    # ==================================================
    # PIPELINE 1
    # Recursive + MiniLM + ChromaDB
    # ==================================================

    chroma_results = (
        chroma_db.similarity_search(
            question,
            k=3
        )
    )

    pipeline1_context = "\n\n".join(
        [
            doc.page_content
            for doc in chroma_results
        ]
    )

    # ==================================================
    # PIPELINE 2
    # Section Based + BGE-M3 + FAISS
    # ==================================================

    query_embedding = (
        bgem3_model.embed_query(
            question
        )
    )

    query_vector = np.array(
        [query_embedding],
        dtype=np.float32
    )

    distances, indices = (
        faiss_index.search(
            query_vector,
            k=3
        )
    )

    pipeline2_context = "\n\n".join(
        [
            section_chunks[idx]
            .page_content
            for idx in indices[0]
        ]
    )

    # ==================================================
    # PIPELINE 1 ANSWER
    # ==================================================

    prompt1 = f"""
You are an Oracle ERP expert.

Answer the question using ONLY the supplied context.

If the answer is not available in the context,
explicitly say:
'I could not find the answer in the retrieved context.'

Context:
{pipeline1_context}

Question:
{question}
"""

    answer1 = (
        llm.invoke(prompt1)
        .content
    )

    # ==================================================
    # PIPELINE 2 ANSWER
    # ==================================================

    prompt2 = f"""
You are an Oracle ERP expert.

Answer the question using ONLY the supplied context.

If the answer is not available in the context,
explicitly say:
'I could not find the answer in the retrieved context.'

Context:
{pipeline2_context}

Question:
{question}
"""

    answer2 = (
        llm.invoke(prompt2)
        .content
    )

    print("\n" + "=" * 100)

    print(
        "PIPELINE 1 ANSWER"
    )

    print("=" * 100)

    print(answer1)

    print("\n" + "=" * 100)

    print(
        "PIPELINE 2 ANSWER"
    )

    print("=" * 100)

    print(answer2)

    # ==================================================
    # CONTINUE?
    # ==================================================

    while True:

        choice = input(
            "\nDo you have any more questions? [Y/N]: "
        ).strip().upper()

        if choice == "Y":

            invalid_attempts = 0

            break

        elif choice == "N":

            print(
                "\nThank you for using Oracle RAG Comparison Lab."
            )

            print(
                "Goodbye!"
            )

            raise SystemExit

        else:

            invalid_attempts += 1

            print(
                "\nPlease enter Y or N."
            )

            if invalid_attempts >= 3:

                print(
                    "\nYou have exceeded the maximum number of invalid responses."
                )

                print(
                    "Exiting application."
                )

                raise SystemExit

# Notebook Conclusion

## End-to-End Inference Complete

Both pipelines now support:

- Context Retrieval
- Qwen3:14B Response Generation
- Interactive Question Handling

## Next Step

Proceed to:

### 11_End_To_End_Comparison.ipynb

Objective:

Compare both RAG pipelines across:

- Chunking
- Embeddings
- Vector Stores
- Retrieval Quality
- Generated Answers
- Overall Performance